In [1]:
import torch
import sys, os
import copy
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/tiffanyyan/Documents/GitHub/QuantBricker
Fixed Income Library is loaded.


In [2]:
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'

axis1 = [1, 3, 5, 7]
x_1 = [3, 4, 5, 6]

interp1 = qfCreate1DInterpolator(axis1, x_1, interp_method, extrap_method)
exponent = interp1.integrate(0., 5., calc_grad=True)

In [3]:
exponent

tensor([21.], dtype=torch.float64, grad_fn=<SumBackward1>)

In [16]:

interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'

axis1 = [1, 3, 5, 7]
x_1 = [3, 4, 5, 6]
axis2 = [1, 2]
x_2 = [4, 6]

# model has two components: each of them has an interpolator
interp1 = qfCreate1DInterpolator(axis1, x_1, interp_method, extrap_method)
interp2 = qfCreate1DInterpolator(axis2, x_2, interp_method, extrap_method)

def discfactor(expiry, interp_type, calc_grad=False):
    exponent = None
    if interp_type == 1:
        exponent = interp1.integrate(0., expiry, calc_grad=calc_grad)
    else:
        exponent = interp2.integrate(0., expiry, calc_grad=calc_grad)
    return torch.exp(-exponent)


params = torch.tensor([2., 3])

v = params[0] * discfactor(2.5, interp_type=1, calc_grad=True) \
    + params[1] * discfactor(1.5, interp_type=2, calc_grad=True)

v.backward()
print(interp1.values.grad)
print(interp2.values.grad)


# # # zero_grad
# # interp1.values.grad.zero_()
# # interp2.values.grad.zero_()


# # Enable grad tracking on both interpolators
# exponent1 = interp1.integrate(0., 2.5, calc_grad=True)
# exponent2 = interp2.integrate(0., 1.5, calc_grad=True)

# df1 = torch.exp(-exponent1.squeeze())
# df2 = torch.exp(-exponent2.squeeze())

# v = params[0] * df1 + params[1] * df2

# # Get gradients without mutating .grad on anything
# grad1, grad2 = torch.autograd.grad(
#     outputs=v,
#     inputs=[interp1.values_, interp2.values_]
# )

# print(grad1)  # d(v) / d(interp1.values_), shape matches interp1.values_
# print(grad2)  # d(v) / d(interp2.values_), shape matches interp2.values_


# # model.colate_grads()
# # for each component, ask the component for the gradient w.r.t their internal parameters


# # go through model component, ask each component for the gradient w.r.t their interanl
# # interp1.values.grad




tensor([-0.0002, -0.0004,  0.0000, -0.0000], dtype=torch.float64)
tensor([-0.0027, -0.0014], dtype=torch.float64)


In [17]:
v1 = - params[0] * discfactor(2.5, interp_type=1, calc_grad=True) \
    - params[1] * discfactor(1.5, interp_type=2, calc_grad=True)

v1.backward()
print(interp1.values.grad)
print(interp2.values.grad)

tensor([0., 0., 0., 0.], dtype=torch.float64)
tensor([0., 0.], dtype=torch.float64)


In [ ]:
tensor([-0.0002, -0.0004,  0.0000, -0.0000], dtype=torch.float64)
tensor([-0.0027, -0.0014], dtype=torch.float64)

In [ ]:

def my_func(x, calc_grad=False):
    a=torch.tensor(1.0,requires_grad=calc_grad)
    b=torch.tensor(2.0,requires_grad=calc_grad)
    return a * x + b


def my_func_grad(x):
    return my_func(x, calc_grad=True).backward()
    


In [ ]:

def my_func(x, calc_grad=False):
    
    a=torch.tensor(1.0,requires_grad=calc_grad)
    b=torch.tensor(2.0,requires_grad=calc_grad)
    return a * x + b


def my_func_grad(x):
    return my_func(x, calc_grad=True).backward()
    


In [9]:
input_x = torch.tensor(3.0) 
y = my_func(input_x)
y

tensor(5.)

In [5]:
input_x = torch.tensor(3.0) 
y = my_func(input_x)
input_x_1 = torch.tensor(4.0) 
y_1 = my_func(input_x_1)

In [7]:
x_1 = torch.tensor([0.05, 0.06], requires_grad=True)
x_2 = torch.tensor([0.04, 0.03], requires_grad=True)
params = torch.tensor([1., 1.], requires_grad=False)   # <-- add this

df1 = torch.exp(-torch.dot(params, x_1))
df2 = torch.exp(-torch.dot(params, x_2))
v = 3 * df1 + 4 * df2
v.backward()                                       

print(x_1.grad)
print(x_2.grad)


tensor([-2.6875, -2.6875])
tensor([-3.7296, -3.7296])


In [31]:
from torch.autograd.functional import jacobian

x = torch.tensor([0.05, 0.06], requires_grad=True)
params = torch.tensor([[1., 1.], [2, 2]])
dfs = torch.exp(torch.matmul(params, x))
jacobian(lambda x: torch.exp(torch.matmul(params, x)), x)

tensor([[1.1163, 1.1163],
        [2.4922, 2.4922]])

In [34]:
x = torch.tensor([0.05, 0.06], requires_grad=True)
params = torch.tensor([[1., 1.], [2, 2]])
dfs = torch.exp(torch.matmul(params, x))

# Gradient of dfs[0] w.r.t. x
grad0 = torch.autograd.grad(dfs, x, grad_outputs=torch.tensor([1., 0.]), retain_graph=True)[0]

# Gradient of dfs[1] w.r.t. x
grad1 = torch.autograd.grad(dfs, x, grad_outputs=torch.tensor([0., 1.]))[0]
grad0
grad1

tensor([2.4922, 2.4922])

In [ ]:

interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'

axis1 = [1, 3, 5, 7]
x_1 = [3, 4, 5, 6]
axis2 = [1, 2]
x_2 = [4, 6]

# model has two components: each of them has an interpolator
interp1 = qfCreate1DInterpolator(axis1, x_1, interp_method, extrap_method)
interp2 = qfCreate1DInterpolator(axis2, x_2, interp_method, extrap_method)

def discfactor(expiry, interp_type, calc_grad=False):
    exponent = None
    if interp_type == 1:
        exponent = interp1.integrate(0., expiry, calc_grad=calc_grad)
    else:
        exponent = interp2.integrate(0., expiry, calc_grad=calc_grad)
    return torch.exp(-exponent)


params = torch.tensor([2., 3])

v = params[0] * discfactor(2.5, interp_type=1, calc_grad=True) \
    + params[1] * discfactor(1.5, interp_type=2, calc_grad=True)


v.backward()
print(x_1.grad)
print(x_2.grad)

v1 = params[0] * discfactor(2.5, interp_type=1, calc_grad=True) \
    + params[1] * discfactor(1.5, interp_type=2, calc_grad=True)

v1.backward()
print(x_1.grad)
print(x_2.grad)





# model.colate_grads()
# for each component, ask the component for the gradient w.r.t their internal parameters


# go through model component, ask each component for the gradient w.r.t their interanl
interp1.values.grad




AttributeError: 'numpy.ndarray' object has no attribute 'grad'

In [ ]:
v = params[0] * discfactor(2.5, interp_type=1, calc_grad=True) \
    + params[1] * discfactor(1.5, interp_type=2, calc_grad=True)

In [40]:
exponent = interp_1d.integrate([0, 0], [2, 4], calc_grad=True)
dfs = torch.exp(-exponent)
dfs


tensor([9.1188e-04, 1.1254e-07], dtype=torch.float64, grad_fn=<ExpBackward0>)

In [ ]:
exponent = interp_1d.integrate([0, 0], [2, 4], calc_grad=True)
dfs = torch.exp(-exponent)

values_tensor = interp_1d._values_tensor  # the leaf with requires_grad=True

# Jacobian: row i = d(dfs[i]) / d(values_tensor)
grad0 = torch.autograd.grad(dfs, values_tensor, grad_outputs=torch.tensor([1., 0.]), retain_graph=True)[0]
grad1 = torch.autograd.grad(dfs, values_tensor, grad_outputs=torch.tensor([0., 1.]))[0]

J = torch.stack([grad0, grad1])  # shape: (2, N) where N = len(values_tensor)


RuntimeError: grad can be implicitly created only for scalar outputs